In [1]:
pip install nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install numpy pandas scikit-learn requests beautifulsoup4 nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [4]:
import numpy as np
import pandas as pd
import re

class BooleanRetrieval:
    def __init__(self):
        self.index = {}
        self.documents_matrix = None
        self.documents = {}

    def index_document(self, doc_id, text):
        self.documents[doc_id] = text
        terms = text.lower().split()
        print(f"Document {doc_id}:", terms)

        for term in terms:
            if term not in self.index:
                self.index[term] = set()
            self.index[term].add(doc_id)

    def create_documents_matrix(self): # Added this function as a method within the class
        terms = list(self.index.keys())
        num_docs = len(self.documents)
        num_terms = len(terms)
        self.documents_matrix = np.zeros((num_docs, num_terms), dtype=int)

        for i, (doc_id, text) in enumerate(self.documents.items()):
            doc_terms = text.lower().split()
            for term in doc_terms:
                if term in self.index:
                    term_id = terms.index(term)
                    self.documents_matrix[i, term_id] = 1

    def print_documents_matrix_table(self): # Added this function as a method within the class
        df = pd.DataFrame(self.documents_matrix, columns=list(self.index.keys()), index=self.documents.keys())
        print("\nDocument-Term Matrix:")
        print(df)

    def print_all_terms(self): # Added this function as a method within the class
        print("\nAll indexed terms:")
        print(list(self.index.keys()))

    def boolean_search(self, query):
        # Preprocessing
        query = query.lower()
        query = re.sub(r'\band\b', '&', query)
        query = re.sub(r'\bor\b', '|', query)
        query = re.sub(r'\bnot\b', '-', query)

        # Replace terms with document sets
        tokens = re.findall(r'\w+|[&|\-()]', query)
        processed_tokens = []
        for token in tokens:
            if token.isalpha():
                docs = self.index.get(token, set())
                processed_tokens.append(f"set({docs})")
            else:
                processed_tokens.append(token)

        # Evaluate the boolean expression
        final_expr = " ".join(processed_tokens)
        try:
            result_set = eval(final_expr)
            return result_set
        except:
            print("Invalid query syntax.")
            return set()

if __name__ == "__main__":
    indexer = BooleanRetrieval()

    documents = {
        1: "Python is a programming language",
        2: "Information retrieval deals with finding information",
        3: "Boolean models are used in information retrieval"
    }

    for doc_id, text in documents.items():
        indexer.index_document(doc_id, text)

    indexer.create_documents_matrix()
    indexer.print_documents_matrix_table()
    indexer.print_all_terms()

    while True:
        query = input("\nEnter your boolean query (or 'exit' to quit): ")
        if query.lower() == 'exit':
            break
        results = indexer.boolean_search(query)
        if results:
            print(f"Results for '{query}': Documents {sorted(results)}")
        else:
            print("No results found for the query.")

Document 1: ['python', 'is', 'a', 'programming', 'language']
Document 2: ['information', 'retrieval', 'deals', 'with', 'finding', 'information']
Document 3: ['boolean', 'models', 'are', 'used', 'in', 'information', 'retrieval']

Document-Term Matrix:
   python  is  a  programming  language  information  retrieval  deals  with  \
1       1   1  1            1         1            0          0      0     0   
2       0   0  0            0         0            1          1      1     1   
3       0   0  0            0         0            1          1      0     0   

   finding  boolean  models  are  used  in  
1        0        0       0    0     0   0  
2        1        0       0    0     0   0  
3        0        1       1    1     1   1  

All indexed terms:
['python', 'is', 'a', 'programming', 'language', 'information', 'retrieval', 'deals', 'with', 'finding', 'boolean', 'models', 'are', 'used', 'in']

Enter your boolean query (or 'exit' to quit): python and programming
Results for